# ⏩ Fluxos de Trabalho Sequenciais com Modelos GitHub (Python)

## 📋 Tutorial Avançado de Processamento Sequencial

Este notebook demonstra **padrões de fluxos de trabalho sequenciais** utilizando o Microsoft Agent Framework. Vai aprender como construir pipelines de processamento sofisticados em várias etapas onde os agentes executam por ordem específica, transmitindo dados e contexto entre as fases.

## 🎯 Objetivos de Aprendizagem

### 🔄 **Padrões de Processamento Sequencial**
- **Design de Workflow Linear**: Criar pipelines de processamento passo a passo
- **Gestão de Fluxo de Dados**: Passar informação entre agentes sequenciais
- **Processamento com Estágios e Portões**: Implementar pontos de verificação e fases de validação
- **Monitorização do Progresso**: Acompanhar a execução do workflow e os resultados intermédios

### 🏗️ **Arquitetura de Pipeline Empresarial**
- **Modelação de Processos de Negócio**: Mapear processos reais para fluxos de agentes
- **Garantia de Qualidade**: Validação e revisão em múltiplas fases
- **Processamento de Documentos**: Análise e transformação sequencial de documentos
- **Produção de Conteúdo**: Workflows editoriais com fases de revisão e aprovação

### 📊 **Funcionalidades Avançadas de Workflow**
- **Preservação de Contexto**: Manter estado ao longo das fases do workflow
- **Propagação de Erros**: Gerir falhas no processamento sequencial
- **Otimização de Desempenho**: Padrões eficientes de execução sequencial
- **Audit Trails**: Registo completo das operações sequenciais

## ⚙️ Pré-requisitos e Configuração

### 📦 **Dependências**
```bash
pip install agent-framework-core -U
```

### 🔑 **Configuração**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

## 🏢 **Casos de Uso Empresariais de Workflow Sequencial**

### 📝 **Pipeline de Processamento de Documentos**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Workflow de Garantia de Qualidade**
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Pipeline de Produção de Conteúdo**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Automatização de Processos de Negócio**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Princípios de Design de Workflow Sequencial**

- **🔗 Progressão Linear**: Cada fase depende da saída da fase anterior
- **📋 Gestão de Estado**: Preservar contexto e dados em todas as fases
- **🛡️ Tratamento de Erros**: Gestão elegante de falhas em qualquer fase
- **📊 Monitorização do Progresso**: Acompanhar conclusão e desempenho em cada estágio
- **🔄 Reutilização de Estágios**: Desenhar componentes de workflow reutilizáveis

Vamos construir workflows de processamento sequenciais sofisticados! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = await provider.create_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = await provider.create_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = await provider.create_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
